# planejamento de gank no league of legends

ambiente do lol com bfs

  usos de IA !! :                                                            
-> o grafo foi feito por mim no graphlab e transformado em codigo pelo chatgpt para ambientar a aplicação                          
->tambem usei o chatgpt para me ajudar nos botoes de prox e anterior no passo a passo da bfs
       


---


  OBSERVACOES:                                                                       
*na ultima etapa, a rota encontrada aparece em amarelo, com os nos em verde p dar destaque ao cmainho escolhido pela bfs                                           
*o botao= ver rota vai diretamente para o resultado final

-O mapa eh o grafo que fiz no graphlab e foi transformaod em codigo: os nós cinzas são caminhos por onde o jungler pode andar, as torres, monstros e objetivos servem como ref, a torre(tanto aliada quanto inimiga) e o bjetivo sendo locais de gank

-O time escolhido eh o do jungler, o lado do gank indica quais estruturas queremos gankar (azuis ou vermelhas)

-T1 e T2 destruidas se referem ao lado do gank, indicando se o destino sera proximo da T1, T2 ou T3


---


-Visualizacao da BFS:                                                            
  -> Verde-agua: nós descobertos durante a busca                                 
  -> Cinza: nós ainda nao descobertos                                           
  -> Amarelo: caminho final escolhido                                            
  -> Verde: nós que fazem parte do caminho final

-Sentinelas e visao:                                                              
  -> Circulo vermelho: região sob visao inimiga  
  -> Círculo azul: regiao sob visao aliada  
  -> Losangos: posicoes das sentinelas

-Quando ativada a opção de evitar visão inimiga, a BFS desconsidera os nós vigiados


In [1]:
# dados do mapa que foi feito por mim no graphlab e trazido apra o codigo pelo chatgpt
# cada no tem id, nome, x, y e cor
# apenas as bolinhas cinzas representam caminhos por onde o jungler anda

import matplotlib.pyplot as plt

cor_caminho = "#64748b"

nos_mapa = [
    (1, 'nexus', -614.1127428636888, 374.332317333043, '#3b82f6'),
    (7, 'torre nexus', -587.1153176225602, 306.9630370166, '#3b82f6'),
    (8, 'torre nexus', -538.166471115437, 375.7681744971691, '#3b82f6'),
    (9, 'inibidor bot', -393.02537497164394, 402.548561272146, '#3b82f6'),
    (10, 'inimidor top', -627.6672843798226, 128.519847360114, '#3b82f6'),
    (11, 'inibidor mid', -481.7848496615724, 226.49250040828306, '#3b82f6'),
    (12, 't3 top', -618.2483433684378, 74.97777819049465, '#3b82f6'),
    (13, 't3 mid', -439.8294316077204, 193.9697458928417, '#3b82f6'),
    (14, 't3 bot', -332.8149435592933, 405.6543270195007, '#3b82f6'),
    (15, 't2 top', -603.7044161015741, -63.93729127447058, '#3b82f6'),
    (16, 't1 top', -516.7948978088266, -417.3416759136368, '#3b82f6'),
    (17, 'sapo', -559.0642287608242, -190.5801165151904, '#14b8a6'),
    (18, 'azul', -452.15500029547184, -168.59910290409465, '#14b8a6'),
    (19, 'lobos', -384.65114000831636, -54.01063696049293, '#14b8a6'),
    (20, 't2 mid', -327.7504137386943, 75.75502091400149, '#3b82f6'),
    (21, 't1 mid', -164.495889222047, -53.331435821241755, '#3b82f6'),
    (22, 'covil barao /arauto', -202.44668804953847, -374.70882070385494, '#8b5cf6'),
    (23, 'aronguejo', -19.5211345888903, -54.93692622197913, '#84cc16'),
    (24, 'galinha', -164.56270321173486, 5.51507508504306, '#14b8a6'),
    (25, 'vermelho', -70.73871597920284, 157.61513244636458, '#14b8a6'),
    (26, 't2 bot', -154.10514833356328, 399.88406791413627, '#3b82f6'),
    (27, 'pedra', -69.10248037389648, 276.30941048528024, '#14b8a6'),
    (28, 't1 bot', 89.90450694556452, 378.6728488951227, '#3b82f6'),
    (29, 'covil dragao', 112.59773147666034, 161.65927939855598, '#8b5cf6'),
    (30, 'aronguejo', -158.19689537357067, -185.74546751321634, '#84cc16'),
    (31, 't1 mid', 74.37645192671397, -228.414843074396, '#f43f5e'),
    (32, 'galinha', 25.388027247606622, -267.7538477655278, '#f97316'),
    (33, 'vermelho', -51.161685867433924, -375.8442065811342, '#f97316'),
    (34, 't1 top', -349.62989358533565, -552.1016663599404, '#f43f5e'),
    (35, 't1 bot', 424.14436984023155, 108.65149222368062, '#f43f5e'),
    (36, 'sapo', 388.89326847934615, -0.30976099025474646, '#f97316'),
    (37, 'azul', 253.60917123635534, -37.24774579228797, '#f97316'),
    (38, 'lobos', 227.3563142170318, -179.24845937093914, '#f97316'),
    (39, 't2 mid', 218.47347330057758, -333.8680933722524, '#f43f5e'),
    (40, 't3 mid', 339.33584731917585, -453.65124534723395, '#f43f5e'),
    (41, 'inibidor mid', 379.3135317088187, -494.770385849595, '#f43f5e'),
    (42, 'inibidor bot', 512.0361125822071, -400.2562252059626, '#f43f5e'),
    (43, 'inibidor top', 249.9805829717606, -611.3803349713194, '#f43f5e'),
    (44, 't3 top', 199.79285336161516, -605.5967196965329, '#f43f5e'),
    (45, 't3 bot', 504.1488368409826, -346.1061077083216, '#f43f5e'),
    (46, 't2 top', 13.58976597554782, -612.7837999277804, '#f43f5e'),
    (47, 'pedra', -91.43853907261845, -469.6147916206557, '#f97316'),
    (48, 't2 bot', 469.16095437813493, -127.1590681619118, '#f43f5e'),
    (49, 'nexus', 487.1265071344207, -606.8073806786804, '#f43f5e'),
    (50, 'torre nexus', 486.95688628384835, -540.0101149147783, '#f43f5e'),
    (51, 'torre nexus', 419.8278648982739, -604.7233266831618, '#f43f5e'),
    (56, 'base_top', -613.2624656777551, 7.842258676286042, '#64748b'),
    (57, 'base_mid', -393.0444156258469, 152.78659619583692, '#64748b'),
    (58, 'bot_a01', -317.747507244179, 179.28420763483683, '#64748b'),
    (59, 'bot_a02', -256.5481427990343, 139.9633299803313, '#64748b'),
    (60, 'bot_a03', -164.43951120007506, 81.58092297934336, '#64748b'),
    (61, 'bot_a04', -258.5811812686274, 27.119445669399283, '#64748b'),
    (62, 'bot_a05', -38.880676072641975, 28.907138234350306, '#64748b'),
    (64, 'bot_a07', -210.57796714691224, 249.37789649042017, '#64748b'),
    (65, 'bot_a06', -145.55270164114677, 321.8452514013335, '#64748b'),
    (67, 'bot_a09', -220.19529218409818, 399.6753598578772, '#64748b'),
    (70, 'bot_a12', 6.031180486361242, 168.84830972128628, '#64748b'),
    (71, 'bot_a13', -76.25932369971675, 406.4164817709459, '#64748b'),
    (72, 'bot_a14', -41.40540004973472, 336.4796078416591, '#64748b'),
    (73, 'bot_a15', 30.778291567281556, 395.3999303154836, '#64748b'),
    (74, 'bot_a16', 107.53127873084914, 305.9114991313745, '#64748b'),
    (76, 'bot_a18', 159.60942315675408, 379.9675563971495, '#64748b'),
    (78, 'bot_a19', -96.16158931587094, 230.87187781010851, '#64748b'),
    (79, 'bot_a20', -145.53048853710146, 168.3188738599726, '#64748b'),
    (80, 'bot_a11', 168.77559753373805, 268.16705004237434, '#64748b'),
    (81, 'bot_a21', 242.5320942746601, 325.1748886085301, '#64748b'),
    (83, 'mid_a01', -114.29167066146418, -89.58665934552967, '#64748b'),
    (84, 'bot_v01', 345.7672181674322, 260.7448923201583, '#64748b'),
    (85, 'bot_v02', 185.46991026642516, 121.47281874626658, '#64748b'),
    (88, 'rio_m01', -47.02181843238294, -143.58753477677885, '#64748b'),
    (91, 'top_a03', -524.9172719891112, -92.83287386943633, '#64748b'),
    (93, 'top_a05', -595.6267596636935, -139.28915617550913, '#64748b'),
    (94, 'top_a01', -502.7178664327534, 24.136374533668075, '#64748b'),
    (96, 'top_a04', -437.74027734591, -91.94992487437729, '#64748b'),
    (97, 'top_a06', -420.1327879121001, 16.390758851739758, '#64748b'),
    (98, 'top_a07', -323.04617563989314, -27.53812772513908, '#64748b'),
    (99, 'top_a08', -329.03110620020726, -114.73292376519184, '#64748b'),
    (101, 'bot_a10', -356.80246583409945, -204.08092679541133, '#64748b'),
    (102, 'top_a09', -219.387456482291, -126.00592145626769, '#64748b'),
    (104, 'top_a11', -252.29987360748038, -299.06441939003787, '#64748b'),
    (105, 'top_a12', -349.6059206198472, -388.85722978097084, '#64748b'),
    (106, 'top_a13', -483.80238023983264, -278.9339288360952, '#64748b'),
    (107, 'top_a14', -486.4226347950143, -364.5531761026848, '#64748b'),
    (108, 'top_a15', -565.2161454145961, -324.52701447620785, '#64748b'),
    (109, 'top_a16', -443.4728132689318, -220.45758080816972, '#64748b'),
    (110, 'top_a17', -505.93208958096284, -183.10221613793263, '#64748b'),
    (111, 'top_a18', -545.4351291652504, -259.91316753620777, '#64748b'),
    (112, 'top_a19', -463.64137148936334, -492.225676715241, '#64748b'),
    (114, 'top_v02', -256.42725872391236, -447.895628900038, '#64748b'),
    (115, 'top_v03', -142.5886058741786, -427.923408618625, '#64748b'),
    (116, 'top_v04', -133.83002246330278, -351.4608553459774, '#64748b'),
    (117, 'rio_s01', -65.358956903349, -300.34875738764583, '#64748b'),
    (118, 'top_l01', -293.8388480391018, -587.4494669249303, '#64748b'),
    (119, 'top_v01', -26.185233091987772, -438.875611790723, '#64748b'),
    (120, 'rio_s02', 42.13809855208121, -393.35820017605863, '#64748b'),
    (121, 'rio_s03', 94.05778838923958, -316.3285411613607, '#64748b'),
    (122, 'mid_v02', 171.14967762502582, -289.9548771128252, '#64748b'),
    (123, 'mid_v01', 29.63407171000631, -194.41833633738358, '#64748b'),
    (124, 'top_v10', -31.60220566074869, -512.6336445183133, '#64748b'),
    (125, 'top_v11', 128.41057767789906, -399.06470689640946, '#64748b'),
    (126, 'top_v12', -142.6199130402439, -514.8457622211856, '#64748b'),
    (127, 'rio_i01', 76.60517552029803, -2.8901722399065424, '#64748b'),
    (128, 'base_a_top01', -631.684000944665, 203.01572508419378, '#64748b'),
    (129, 'base_a_centro01', -518.5701908299587, 273.04212379150306, '#64748b'),
    (130, 'base_a_bot01', -465.5739078790092, 414.5866215568967, '#64748b'),
    (131, 'base_a_mid01', -367.7851564876841, 265.01829443207043, '#64748b'),
    (132, 'base_a_top02', -533.884456646964, 123.30176295360968, '#64748b'),
    (133, 'nascimento', -685.125017844865, 415.84453025693574, '#64748b'),
    (134, 'bot_v03', 407.47087753237935, 154.61901986722944, '#64748b'),
    (135, 'bot_v04', 351.8621841205479, 123.18801924315075, '#64748b'),
    (136, 'bot_v05', 376.8458000012258, 53.878633251592646, '#64748b'),
    (137, 'bot_v06', 324.4607989610947, 190.88555904885868, '#64748b'),
    (138, 'bot_v07', 282.5527981289898, 102.2340188270983, '#64748b'),
    (140, 'bot_v08', 309.14826019551793, -13.012983461190174, '#64748b'),
    (141, 'bot_v09', 267.24025936341303, 21.64170953458888, '#64748b'),
    (142, 'mid_v03', 167.30579584070136, -32.355137691392436, '#64748b'),
    (143, 'mid_v04', 175.36502676995235, -108.91783151927639, '#64748b'),
    (144, 'mid_v05', 381.68133855877653, -290.2505274274226, '#64748b'),
    (145, 'mid_v06', 330.10226061157044, -101.66452368295054, '#64748b'),
    (146, 'mid_v07', 91.54902510574252, -110.52967770512657, '#64748b'),
    (147, 'mid_v08', 149.5754877963493, -204.0167564844375, '#64748b'),
    (148, 'bot_v10', 456.63218620081017, -52.50321501451979, '#64748b'),
    (149, 'bot_v11', 501.76387940461547, -260.43137298919413, '#64748b'),
    (150, 'mid_v09', 259.9869515270872, -269.29652701137013, '#64748b'),
    (151, 'mid_v10', 303.5067985450424, -197.5693717410368, '#64748b'),
    (153, 'top_v13', 272.8817210138887, -388.5731447642841, '#64748b'),
    (155, 'top_v14', 205.99010430110587, -444.1818381761156, '#64748b'),
    (156, 'top_v15', 74.62464015431551, -607.7842260399096, '#64748b'),
    (157, 'top_v16', -73.70600135169865, -608.9043598884454, '#64748b'),
    (159, 'top_v17', 81.87794799064133, -527.1919167474002, '#64748b'),
    (160, 'top_v18', 143.12810305294846, -604.5605336682094, '#64748b'),
    (161, 'base_v01', 330.58516033350884, -617.5075050489672, '#64748b'),
    (162, 'base_v02', 538.8363416791698, -651.3040730578648, '#64748b'),
    (163, 'base_v03', 541.254110957945, -474.8069157072692, '#64748b'),
    (164, 'base_v04', 425.20118557673146, -528.8037629332505, '#64748b'),
    (165, 'sentinela_azul_1', -272.2998736074804, -319.06441939003787, '#38a9ed'),
    (166, 'sentinela_azul_2', -134.2916706614642, -66.58665934552967, '#38a9ed'),
    (167, 'sentinela_azul_3', -60.880676072641975, 50.907138234350306, '#38a9ed'),
    (168, 'sentinela_azul_4', -74.16158931587094, 212.87187781010851, '#38a9ed'),
    (169, 'sentinela_vermelho_1', 118.05778838923958, -336.3285411613607, '#ef6380'),
    (170, 'sentinela_vermelho_2', 194.14967762502582, -267.9548771128252, '#ef6380'),
    (171, 'sentinela_vermelho_3', 186.30579584070136, -54.355137691392436, '#ef6380'),
    (172, 'sentinela_vermelho_4', 305.5527981289898, 125.2340188270983, '#ef6380'),
]

ligacoes = [
    (56, 57),
    (57, 58),
    (58, 64),
    (58, 59),
    (64, 65),
    (65, 67),
    (59, 60),
    (59, 61),
    (60, 62),
    (72, 73),
    (72, 74),
    (74, 76),
    (74, 70),
    (70, 78),
    (64, 78),
    (78, 79),
    (65, 78),
    (79, 60),
    (62, 70),
    (74, 80),
    (80, 81),
    (62, 83),
    (85, 84),
    (83, 88),
    (56, 94),
    (57, 94),
    (91, 93),
    (91, 96),
    (96, 97),
    (97, 98),
    (98, 61),
    (96, 99),
    (99, 98),
    (57, 61),
    (60, 61),
    (61, 83),
    (99, 101),
    (99, 102),
    (102, 83),
    (101, 104),
    (104, 105),
    (105, 106),
    (106, 107),
    (107, 108),
    (106, 109),
    (109, 110),
    (110, 91),
    (110, 111),
    (111, 108),
    (93, 108),
    (56, 93),
    (105, 112),
    (108, 112),
    (105, 114),
    (114, 115),
    (115, 116),
    (104, 88),
    (114, 118),
    (115, 119),
    (116, 117),
    (117, 120),
    (120, 121),
    (121, 122),
    (117, 123),
    (117, 88),
    (88, 123),
    (123, 122),
    (119, 120),
    (119, 124),
    (121, 125),
    (124, 126),
    (114, 126),
    (126, 118),
    (80, 85),
    (62, 127),
    (127, 88),
    (85, 127),
    (78, 25),
    (60, 24),
    (65, 72),
    (65, 71),
    (72, 27),
    (85, 29),
    (97, 19),
    (110, 17),
    (110, 18),
    (109, 18),
    (111, 17),
    (111, 107),
    (104, 22),
    (71, 73),
    (73, 76),
    (76, 81),
    (81, 84),
    (112, 118),
    (130, 131),
    (131, 57),
    (130, 129),
    (129, 128),
    (128, 132),
    (132, 57),
    (132, 56),
    (94, 91),
    (67, 131),
    (133, 129),
    (133, 130),
    (133, 128),
    (130, 67),
    (128, 56),
    (129, 57),
    (129, 131),
    (129, 132),
    (67, 71),
    (138, 85),
    (134, 135),
    (135, 137),
    (137, 138),
    (135, 136),
    (136, 36),
    (36, 140),
    (140, 37),
    (138, 141),
    (141, 140),
    (142, 146),
    (146, 123),
    (142, 127),
    (142, 143),
    (143, 145),
    (145, 140),
    (145, 148),
    (145, 144),
    (144, 149),
    (143, 147),
    (147, 122),
    (122, 150),
    (150, 151),
    (151, 145),
    (151, 38),
    (144, 153),
    (125, 155),
    (155, 153),
    (124, 157),
    (118, 157),
    (117, 121),
    (125, 159),
    (124, 159),
    (159, 156),
    (155, 160),
    (160, 156),
    (162, 161),
    (161, 160),
    (162, 163),
    (162, 164),
    (161, 164),
    (163, 164),
    (163, 149),
    (157, 156),
    (149, 148),
    (148, 134),
    (134, 84),
    (164, 153),
    (153, 122),
    (126, 47),
    (117, 32),
    (119, 33),
    (102, 30),
    (88, 30),
    (62, 23),
    (127, 23),
    (83, 23),
    (88, 23),
    (146, 23),
    (165, 104),
    (166, 83),
    (167, 62),
    (168, 78),
    (169, 121),
    (170, 122),
    (171, 142),
    (172, 138),
]


In [2]:
# configuracoes do gank
# os pontos de gank sao bolinhas cinzas proximas das rotas (optei por ser a bolinha cinza "na frente" da torre ou objetivo)

# cada fase do gank tem destinos diferentes
# t1 de pe: aproxima da t1
# t1 destruida: aproxima da t2
# t1 e t2 destruidas: aproxima da t3
# os destinos sao sempre nos cinzas, nunca as torres coloridas
pontos_gank = {
    "azul": {
        "top": {"t1": ["top_a19", "top_a15"], "t2": ["top_a05"], "t3": ["base_top"]},
        "mid": {"t1": ["mid_a01", "top_a09"], "t2": ["bot_a04"], "t3": ["base_mid"]},
        "bot": {"t1": ["bot_a16", "bot_a18", "bot_a15"], "t2": ["bot_a13"], "t3": ["bot_a09"]}
    },
    "vermelho": {
        "top": {"t1": ["top_l01"], "t2": ["top_v15", "top_v16"], "t3": ["top_v18"]},
        "mid": {"t1": ["mid_v01", "mid_v08"], "t2": ["mid_v02", "mid_v09"], "t3": ["top_v13"]},
        "bot": {"t1": ["bot_v03", "bot_v04"], "t2": ["bot_v10"], "t3": ["bot_v11"]}
    }
}

objetivos = {"superior": 22, "dragao": 29}

# as sentinelas ficam fixas no mapa
# elas mostram o no escolhido e seus vizinhos diretos
# a cor azul e vermelha indica o time dono da sentinela
posicoes_sentinelas = [
    ('azul', 'top_a11', 'sentinela_azul_1'),  # entrada superior e barao
    ('azul', 'mid_a01', 'sentinela_azul_2'),  # acesso ao meio
    ('azul', 'bot_a05', 'sentinela_azul_3'),  # entrada inferior e dragao
    ('azul', 'bot_a19', 'sentinela_azul_4'),  # selva inferior azul
    ('vermelho', 'rio_s03', 'sentinela_vermelho_1'),  # entrada superior vermelha
    ('vermelho', 'mid_v02', 'sentinela_vermelho_2'),  # acesso ao meio vermelho
    ('vermelho', 'mid_v03', 'sentinela_vermelho_3'),  # selva inferior vermelha
    ('vermelho', 'bot_v07', 'sentinela_vermelho_4'),  # aproximacao inferior vermelha
]



def carregar_ambiente():
    nos = {}
    nomes = {}
    grafo = {}

    for identificador, nome, x, y, cor in nos_mapa:
        nos[identificador] = {"id": identificador, "label": nome,
                               "x": x, "y": y, "color": cor}
        if cor == cor_caminho:
            nomes[nome] = identificador
            grafo[identificador] = []

    referencias = []
    for a, b in ligacoes:
        if a in grafo and b in grafo:
            grafo[a].append(b)
            grafo[b].append(a)
        else:
            referencias.append((a, b))

    return {"nos": nos, "nomes": nomes, "grafo": grafo,
            "referencias": referencias}


def nomes_dos_nos(ambiente, lista):
    nomes = []
    for identificador in lista:
        nomes.append(ambiente["nos"][identificador]["label"])
    return nomes


def destinos_disponiveis(ambiente, modo, lado, rota, t1, t2, objetivo):
    if modo == "objetivo":
        covil = objetivos[objetivo]
        destinos = []
        for a, b in ambiente["referencias"]:
            if a == covil and b in ambiente["grafo"]:
                destinos.append(b)
            if b == covil and a in ambiente["grafo"]:
                destinos.append(a)
        return set(destinos)

    if t2 and not t1:
        raise ValueError("a t1 deve ser destruida antes da t2")

    # com t1 e t2 destruidas, o destino e a regiao da t3
    torre = "t1"
    if t1:
        torre = "t2"
    if t2:
        torre = "t3"
    nomes = pontos_gank[lado][rota][torre]

    destinos = set()
    for nome in nomes:
        destinos.add(ambiente["nomes"][nome])
    return destinos


def encontrar_torre(ambiente, lado, rota, t1, t2):
    # acha a estrutura colorida referente ao destino do gank
    nivel = "t1"
    if t1:
        nivel = "t2"
    if t2:
        nivel = "t3"

    nome = nivel + " " + rota
    cor = "#3b82f6" if lado == "azul" else "#f43f5e"
    for identificador, no in ambiente["nos"].items():
        if no["label"] == nome and no["color"] == cor:
            return identificador
    return None


def criar_sentinela(time, posicao, cobertura=None):
    if cobertura is None:
        cobertura = [posicao]
    return {"time": time, "posicao": posicao, "cobertura": cobertura}


def sentinelas_fixas(ambiente):
    sentinelas = []
    for time, posicao, marcador in posicoes_sentinelas:
        identificador = ambiente["nomes"][posicao]
        cobertura = [posicao]
        for vizinho in ambiente["grafo"][identificador]:
            cobertura.append(ambiente["nos"][vizinho]["label"])
        sentinela = criar_sentinela(time, posicao, cobertura)
        sentinela["marcador"] = marcador
        sentinelas.append(sentinela)
    return sentinelas


def criar_cenario(ambiente, inicio="top_a17", time="azul", modo="gank",
                  lado="azul", rota="top", objetivo="superior", t1=False,
                  t2=False, sentinelas=None, evitar_visao=False, bloqueios=None):
    nomes = ambiente["nomes"]
    if inicio not in nomes:
        raise ValueError("o inicio deve ser um no cinza do mapa")

    if sentinelas is None:
        sentinelas = sentinelas_fixas(ambiente)
    bloqueios = bloqueios or []
    visao_aliada = set()
    visao_inimiga = set()

    for sentinela in sentinelas:
        if sentinela["posicao"] not in nomes:
            raise ValueError("posicao de sentinela invalida")
        for nome in sentinela["cobertura"]:
            if nome not in nomes:
                raise ValueError("cobertura de sentinela invalida: " + nome)
            if sentinela["time"] == time:
                visao_aliada.add(nomes[nome])
            else:
                visao_inimiga.add(nomes[nome])

    proibidos = set()
    for nome in bloqueios:
        if nome not in nomes:
            raise ValueError("no bloqueado nao existe: " + nome)
        proibidos.add(nomes[nome])

    if evitar_visao:
        proibidos.update(visao_inimiga)
    if nomes[inicio] in proibidos:
        raise ValueError("o ponto inicial esta bloqueado")

    destinos = destinos_disponiveis(ambiente, modo, lado, rota, t1, t2, objetivo)
    destinos_livres = destinos - proibidos
    torre_alvo = None
    if modo == "gank":
        torre_alvo = encontrar_torre(ambiente, lado, rota, t1, t2)

    return {
        "inicio": nomes[inicio], "time": time, "modo": modo, "lado": lado,
        "rota": rota, "objetivo": objetivo, "t1": t1, "t2": t2,
        "sentinelas": sentinelas, "visao_aliada": visao_aliada,
        "visao_inimiga": visao_inimiga, "bloqueados": proibidos,
        "destinos": destinos, "destinos_livres": destinos_livres,
        "torre_alvo": torre_alvo
    }


def preparar_busca(ambiente, cenario):
    grafo = {}
    proibidos = cenario["bloqueados"]

    for atual, vizinhos in ambiente["grafo"].items():
        if atual not in proibidos:
            grafo[atual] = []
            for vizinho in vizinhos:
                if vizinho not in proibidos:
                    grafo[atual].append(vizinho)

    return grafo, cenario["inicio"], cenario["destinos_livres"]


def validar_caminho(ambiente, cenario, caminho):
    if caminho is None or len(caminho) == 0:
        return False
    if caminho[0] != cenario["inicio"]:
        return False
    if caminho[-1] not in cenario["destinos_livres"]:
        return False

    grafo, inicio, destinos = preparar_busca(ambiente, cenario)
    for i in range(len(caminho) - 1):
        atual = caminho[i]
        proximo = caminho[i + 1]
        if atual not in grafo or proximo not in grafo[atual]:
            return False
    return True


def mostrar_mapa(ambiente, cenario, caminho=None, visitados=None, atual=None, imagem=False):
    # desenha o mapa usando as coordenadas do graphlab - feito com ajuda do chatgpt
    caminho = caminho or []
    visitados = visitados or []
    nos = ambiente["nos"]
    grafo = ambiente["grafo"]
    fig, ax = plt.subplots(figsize=(7.8, 5.0), dpi=85)
    fig.patch.set_facecolor("#0b1022")
    ax.set_facecolor("#0b1022")

    # linhas cinzas sao caminhos do jungle, as outras sao so >>ref<<
    for a, b in ligacoes:
        no_a = nos[a]
        no_b = nos[b]
        cor = "#5e7196" if a in grafo and b in grafo else "#38435b"
        ax.plot([no_a["x"], no_b["x"]], [no_a["y"], no_b["y"]],
                color=cor, linewidth=1.4, zorder=1)

    # destaca o caminho final em amarelo e os nos em verde
    for i in range(len(caminho) - 1):
        a = nos[caminho[i]]
        b = nos[caminho[i + 1]]
        ax.plot([a["x"], b["x"]], [a["y"], b["y"]],
                color="#f9e038", linewidth=5, zorder=3)

    for identificador, no in nos.items():
        cor = no["color"]
        if identificador in visitados:
            cor = "#6fc7ef"
        if identificador in caminho:
            cor = "#a3e635"
        if identificador == atual and identificador not in caminho:
            cor = "#e879a6"
        if identificador == cenario["inicio"]:
            cor = "#16b6d5"

        if no["label"].startswith("sentinela_"):
            ax.scatter(no["x"], no["y"], s=95, marker="D",
                       color=cor, edgecolors="white", linewidths=0.8, zorder=7)
        else:
            ax.scatter(no["x"], no["y"], s=90 if identificador in grafo else 75,
                       color=cor, edgecolors="white", linewidths=0.7, zorder=4)
        if identificador in grafo:
            ax.text(no["x"], no["y"] - 17, no["label"],
                    fontsize=5.5, ha="center", color="white")

    # a torre marcada e uma referencia visual, nao um ponto de passagem
    if cenario.get("torre_alvo") is not None:
        torre = nos[cenario["torre_alvo"]]
        ax.scatter(torre["x"], torre["y"], s=400, facecolors="none",
                   edgecolors="#f9e038", linewidths=2.5, zorder=7)
        ax.text(torre["x"], torre["y"] - 32, torre["label"] + " - alvo",
                fontsize=7, ha="center", color="#f9e038", zorder=8)

    # mostra os destinos, os bloqueios e as regioes q tem visao
    for identificador in cenario["destinos"]:
        no = nos[identificador]
        ax.scatter(no["x"], no["y"], s=270, facecolors="none",
                   edgecolors="#d4a017", linewidths=2, zorder=5)
    for identificador in cenario["bloqueados"]:
        no = nos[identificador]
        ax.scatter(no["x"], no["y"], marker="x", s=120,
                   color="red", linewidths=2, zorder=6)
    for identificador in cenario["visao_inimiga"]:
        no = nos[identificador]
        ax.scatter(no["x"], no["y"], s=380, facecolors="none",
                   edgecolors="#df5962", linewidths=1.3, zorder=3)
    for identificador in cenario["visao_aliada"]:
        no = nos[identificador]
        ax.scatter(no["x"], no["y"], s=340, facecolors="none",
                   edgecolors="#38a9ed", linewidths=1.3, zorder=3)

    if cenario["modo"] == "objetivo":
        no = nos[objetivos[cenario["objetivo"]]]
        ax.scatter(no["x"], no["y"], s=350, facecolors="none",
                   edgecolors="#9c65cc", linewidths=3, zorder=5)

    # losangos azuis e vermelhos indicam as sentinelas fixas
    ax.text(0.02, 0.02, "losango azul = sentinela azul  |  losango rosa = sentinela vermelha",
            transform=ax.transAxes, fontsize=6.0, color="white", zorder=9)
    ax.set_aspect("equal")
    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.92)
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title("mapa oficial atualizado - 143 nos, 180 ligacoes, 8 sentinelas",
                 fontsize=9.5, color="white")
    if imagem:
        from io import BytesIO
        arquivo = BytesIO()
        fig.savefig(arquivo, format="png", dpi=85, facecolor=fig.get_facecolor())
        plt.close(fig)
        return arquivo.getvalue()

    plt.show()
    plt.close(fig)


def mostrar_resumo(ambiente, cenario):
    print("time do jungler:", cenario["time"])
    print("inicio:", nomes_dos_nos(ambiente, [cenario["inicio"]])[0])
    print("modo:", cenario["modo"])
    if cenario["modo"] == "gank":
        print("lado das estruturas do gank:", cenario["lado"], "| rota:", cenario["rota"])
        print("t1 do lado do gank destruida:", "sim" if cenario["t1"] else "nao",
              "| t2 do lado do gank destruida:", "sim" if cenario["t2"] else "nao")
        torre = cenario.get("torre_alvo")
        if torre is not None:
            print("torre de referencia:", ambiente["nos"][torre]["label"], "do lado", cenario["lado"])
    else:
        print("objetivo:", cenario["objetivo"])
    print("destinos:", nomes_dos_nos(ambiente, sorted(cenario["destinos_livres"])))
    print("sentinelas fixas azuis:", [s["posicao"] for s in cenario["sentinelas"] if s["time"] == "azul"])
    print("sentinelas fixas vermelhas:", [s["posicao"] for s in cenario["sentinelas"] if s["time"] == "vermelho"])
    print("visao inimiga:", nomes_dos_nos(ambiente, sorted(cenario["visao_inimiga"])))
    print("bloqueados:", nomes_dos_nos(ambiente, sorted(cenario["bloqueados"])))


In [3]:
# busca em largura
# fifo, a fila guarda os nos que ainda precisam ser analisados

def bfs(grafo, inicio, destinos):
    fila = [inicio]
    visitados = {inicio}
    anteriores = {inicio: None}
    ordem = []
    historico = []

    while len(fila) > 0:
        atual = fila.pop(0)
        ordem.append(atual)

        if atual in destinos:
            caminho = []
            posicao = atual
            while posicao is not None:
                caminho.insert(0, posicao)
                posicao = anteriores[posicao]

            historico.append({"atual": atual, "fila": fila.copy(),
                              "analisados": ordem.copy(), "encontrou": True})
            return caminho, ordem, historico

        for vizinho in grafo[atual]:
            if vizinho not in visitados:
                visitados.add(vizinho)
                anteriores[vizinho] = atual
                fila.append(vizinho)

        historico.append({"atual": atual, "fila": fila.copy(),
                          "analisados": ordem.copy(), "encontrou": False})

    return None, ordem, historico


# da p escolher o cenario
# ou aperta enter p usar o padrao

from IPython.display import display

# confere se as celulas com os dados do mapa novo foram executadas
if len(nos_mapa) != 143 or len(ligacoes) != 180:
    raise RuntimeError("mapa antigo carregado. execute todas as celulas deste notebook")

ambiente = carregar_ambiente()
print("mapa oficial carregado: 143 nos, 180 ligacoes, 89 caminhos cinzas e 8 sentinelas")


padrao = {
    "time": "azul",
    "inicio": "top_a17",
    "modo": "gank",
    "lado": "azul",
    "rota": "top",
    "objetivo": "superior",
    "t1": False,
    "t2": False,
    "evitar_visao": False,
    "bloqueios": ""
}


def perguntar(texto, opcoes, padrao):
    # pergunta ate receber uma opcao valida
    while True:
        print("opcoes:", ", ".join(opcoes))
        resposta = input(texto + " [" + padrao + "]: ").strip().lower()
        if resposta == "":
            return padrao
        if resposta in opcoes:
            return resposta
        print("opcao invalida !!! tente novamente\n")


def perguntar_no(texto, padrao):
    # permite consultar os nomes de todos os nos cinzas
    while True:
        print("digite o nome de um no cinza ou 'lista' para ver todos")
        resposta = input(texto + " [" + padrao + "]: ").strip().lower()
        if resposta == "":
            return padrao
        if resposta == "lista":
            nomes = sorted(ambiente["nomes"])
            for i in range(0, len(nomes), 8):
                print(", ".join(nomes[i:i+8]))
            continue
        if resposta in ambiente["nomes"]:
            return resposta
        print("no nao encontrado !!! tente novamente\n")


def perguntar_nomes(texto, padrao):
    # permite informar varios nos separados por virgula
    while True:
        resposta = input(texto + " [" + padrao + "]: ").strip().lower()
        if resposta == "":
            return padrao
        if resposta == "nenhum":
            return ""
        try:
            ler_nomes(resposta)
            return resposta
        except ValueError as erro:
            print("erro:", erro)


def ler_nomes(texto):
    lista = []
    for parte in texto.split(","):
        nome = parte.strip()
        if nome:
            if nome not in ambiente["nomes"]:
                raise ValueError("no cinza nao encontrado: " + nome)
            if nome not in lista:
                lista.append(nome)
    return lista


def escolher_cenario():
    print("\n\n configuracao do jungler")
    print("1 - executar padrao")
    print("2 - escolher os valores")
    escolha = perguntar("escolha", ["1", "2"], "1")

    config = padrao.copy()
    if escolha == "1":
        return config

    print("\n deixe em branco para manter o valor entre colchetes!!")
    config["time"] = perguntar("time do jungler", ["azul", "vermelho"], config["time"])
    config["inicio"] = perguntar_no("inicio", config["inicio"])
    config["modo"] = perguntar("modo", ["gank", "objetivo"], config["modo"])

    if config["modo"] == "gank":
        config["lado"] = perguntar("lado das estruturas do gank", ["azul", "vermelho"], config["lado"])
        config["rota"] = perguntar("rota", ["top", "mid", "bot"], config["rota"])
        print("as torres destruidas sao do lado escolhido para o gank")
        print("t1 de pe: gank perto da t1 | t1 destruida: gank perto da t2")
        print("t1 e t2 destruidas: gank perto da t3")
        config["t1"] = perguntar("t1 desse lado destruida", ["s", "n"], "n") == "s"
        if config["t1"]:
            config["t2"] = perguntar("t2 desse lado destruida", ["s", "n"], "n") == "s"
    else:
        config["objetivo"] = perguntar("objetivo", ["superior", "dragao"], config["objetivo"])

    print("\nas sentinelas sao fixas no mapa")
    alterar = perguntar("quer alterar visao e bloqueios?", ["s", "n"], "n")
    if alterar == "s":
        config["evitar_visao"] = perguntar("evitar visao inimiga", ["s", "n"], "n") == "s"
        config["bloqueios"] = perguntar_nomes("nos bloqueados, separados por virgula", config["bloqueios"])

    return config


def acompanhar_bfs(ambiente, cenario, historico, caminho):
    # os botoes mudam a etapa sem precisar digitar depois do mapa
    if len(historico) == 0:
        print("a bfs nao analisou nenhum no")
        return

    try:
        import ipywidgets as widgets
        try:
            from google.colab import output
            output.enable_custom_widget_manager()
        except ImportError:
            pass
    except ImportError:
        print("instale ipywidgets para usar os botoes de navegacao")
        print("no colab, execute: %pip install ipywidgets")
        return

    etapa = [0]
    anterior = widgets.Button(description="anterior")
    proximo = widgets.Button(description="proximo")
    ver_rota = widgets.Button(description="ver rota")
    ver_rota.disabled = caminho is None
    titulo = widgets.HTML()
    mapa = widgets.Image(format="png", layout=widgets.Layout(width="670px", max_width="100%"))

    def atualizar():
        numero = etapa[0]
        passo = historico[numero]
        nome_atual = nomes_dos_nos(ambiente, [passo["atual"]])[0]
        nomes_fila = nomes_dos_nos(ambiente, passo["fila"])
        fila = ", ".join(nomes_fila[:6]) or "vazia"
        if len(nomes_fila) > 6:
            fila += " ... (mais " + str(len(nomes_fila) - 6) + ")"
        titulo.value = (
            "<b>etapa " + str(numero + 1) + " de " + str(len(historico)) + "</b>"
            + " | no atual: " + nome_atual
            + "<br>fila: " + fila
            + "<br>nos analisados: " + str(len(passo["analisados"]))
        )
        # mostra a rota somente na ultima etapa da busca
        rota_final = numero == len(historico) - 1 and caminho is not None
        if rota_final:
            nomes_rota = nomes_dos_nos(ambiente, caminho)
            titulo.value += "<br><b>rota encontrada:</b> " + " -> ".join(nomes_rota)
            titulo.value += "<br><span style='color:#a3e635'>linha amarela = caminho escolhido</span>"
        elif passo["encontrou"]:
            titulo.value += "<br>destino encontrado"

        anterior.disabled = numero == 0
        proximo.disabled = numero == len(historico) - 1
        mapa.value = mostrar_mapa(
            ambiente, cenario,
            caminho=caminho if rota_final else None,
            visitados=passo["analisados"],
            atual=passo["atual"], imagem=True
        )

    def avancar(botao):
        if etapa[0] < len(historico) - 1:
            etapa[0] += 1
            atualizar()

    def voltar(botao):
        if etapa[0] > 0:
            etapa[0] -= 1
            atualizar()

    def ir_para_rota(botao):
        etapa[0] = len(historico) - 1
        atualizar()

    anterior.on_click(voltar)
    proximo.on_click(avancar)
    ver_rota.on_click(ir_para_rota)

    # os botoes ficam acima do mapa
    display(widgets.VBox(
        [widgets.HBox([anterior, proximo, ver_rota]), titulo, mapa],
        layout=widgets.Layout(width="690px", max_width="100%")
    ))
    atualizar()


def executar():
    config = escolher_cenario()
    try:
        sentinelas = sentinelas_fixas(ambiente)
        cenario = criar_cenario(
            ambiente,
            inicio=config["inicio"],
            time=config["time"],
            modo=config["modo"],
            lado=config["lado"],
            rota=config["rota"],
            objetivo=config["objetivo"],
            t1=config["t1"],
            t2=config["t2"],
            sentinelas=sentinelas,
            evitar_visao=config["evitar_visao"],
            bloqueios=ler_nomes(config["bloqueios"])
        )

        print("\ncenario escolhido")
        mostrar_resumo(ambiente, cenario)
        grafo, inicio, destinos = preparar_busca(ambiente, cenario)
        caminho, ordem, historico = bfs(grafo, inicio, destinos)

        print("\nresultado da bfs")
        print("quantidade de nos analisados:", len(ordem))
        print("nos analisados:", " -> ".join(nomes_dos_nos(ambiente, ordem)))
        if caminho is None:
            print("nenhum caminho encontrado")
        else:
            print("caminho:", " -> ".join(nomes_dos_nos(ambiente, caminho)))
            print("quantidade de deslocamentos:", len(caminho) - 1)
            print("caminho valido:", "sim" if validar_caminho(ambiente, cenario, caminho) else "nao")

        ver_etapas = perguntar("quer acompanhar os passos da bfs?", ["s", "n"], "s")
        if ver_etapas == "s":
            acompanhar_bfs(ambiente, cenario, historico, caminho)
        else:
            mostrar_mapa(ambiente, cenario, caminho, ordem)
    except (ValueError, KeyError) as erro:
        print("erro:", erro)


executar()


mapa oficial carregado: 143 nos, 180 ligacoes, 89 caminhos cinzas e 8 sentinelas


 configuracao do jungler
1 - executar padrao
2 - escolher os valores
opcoes: 1, 2
escolha [1]: 2

 deixe em branco para manter o valor entre colchetes!!
opcoes: azul, vermelho
time do jungler [azul]: azul
digite o nome de um no cinza ou 'lista' para ver todos
inicio [top_a17]: top_a17
opcoes: gank, objetivo
modo [gank]: gank
opcoes: azul, vermelho
lado das estruturas do gank [azul]: vermelho
opcoes: top, mid, bot
rota [top]: bot
as torres destruidas sao do lado escolhido para o gank
t1 de pe: gank perto da t1 | t1 destruida: gank perto da t2
t1 e t2 destruidas: gank perto da t3
opcoes: s, n
t1 desse lado destruida [n]: s
opcoes: s, n
t2 desse lado destruida [n]: s

as sentinelas sao fixas no mapa
opcoes: s, n
quer alterar visao e bloqueios? [n]: n

cenario escolhido
time do jungler: azul
inicio: top_a17
modo: gank
lado das estruturas do gank: vermelho | rota: bot
t1 do lado do gank destruida: sim | t2 do